# Análise da Variabilidade de Estrelas

Guia de laboratório virtual · Projeto PIBIC-EM

Autores: Ian Daniel Oliveira da Rocha Miranda, Jarlon Gomes Viana e Victor Hugo Peixoto Silva.

Orientador: Thiago Mureebe Carrijo.

Instituto Federal de Goiás · Câmpus Águas Lindas de Goiás.

Material destinado ao ensino médio e à divulgação científica, desenvolvido no projeto “Análise da Variabilidade de Estrelas a partir de Dados da AAVSO”.


## Introdução

Estrelas variáveis são estrelas ou sistemas estelares cujo brilho observado muda com o tempo. Neste guia, iremos nos aprofundar pergunta: como descobrir o tempo de repetição dessa variação a partir de observações feitas em instantes irregulares?

Trabalharemos com séries temporais: tabelas que associam cada instante de observação a uma medida de brilho. Os dados da AAVSO, associação internacional de observadores de estrelas variáveis, permitem investigar objetos reais sem realizar novas observações com um telescópio.

Ao concluir o guia, você deverá conseguir:

- Diferenciar a variação por pulsação de uma variação por eclipse.
- Importar e examinar uma tabela de observações, registrando sua origem e os critérios de seleção.
- Construir uma curva de luz e relacionar magnitude, brilho, período e frequência.
- Usar Lomb–Scargle para encontrar períodos candidatos e avaliar o diagrama de fase.
- Comparar os resultados com um catálogo e comunicar as evidências e limitações da análise.

### Como utilizar este guia

Não é necessário conhecer programação antes de ler o guia. Leia uma etapa, execute o código correspondente e interprete a saída antes de avançar. Uma célula de texto contém as explicações; uma célula de código contém as instruções que o Python executa. Clique na célula e pressione `Shift + Enter`. O símbolo `[*]` indica que o cálculo está em andamento.

Nas células de configuração, altere inicialmente apenas os valores indicados nos comentários. Em Python, `=` atribui um valor a um nome; textos ficam entre aspas; `#` inicia um comentário; `None` significa que uma opção não foi preenchida. Números decimais usam ponto: `0.5`, por exemplo.

Comece com uma estrela e depois repita o procedimento para as outras classes. Registre as respostas em células de texto inseridas abaixo das atividades ou em um caderno. O ajuste de Fourier é um estudo mais aprofundado: primeiro é necessário compreender a curva de luz, o período e a fase.

A simulação abaixo é uma exploração conceitual. Os dados analisados a partir de “Importando os dados” serão observações reais, fornecidas em um arquivo separado. A simulação não envia dados para os cálculos em Python.


## Cefeida

Cefeidas são estrelas pulsantes: suas camadas externas se expandem e se contraem. O raio e a temperatura superficial variam durante o ciclo e, juntos, modificam a energia emitida. As Cefeidas clássicas são estrelas de alta luminosidade, com períodos geralmente da ordem de dias a meses. Sua relação entre período e luminosidade média permite estimar distâncias; a comparação deve considerar a classe da estrela e a faixa de luz observada.

Para uma estrela aproximadamente esférica, a luminosidade total pode ser expressa por

$$L = 4\pi R^2\sigma T_{\mathrm{ef}}^4,$$

em que $R$ é o raio, $T_{\mathrm{ef}}$ é a temperatura efetiva e $\sigma$ é a constante de Stefan–Boltzmann. A temperatura entra elevada à quarta potência: conhecer apenas o raio não basta para prever o brilho. Essa expressão descreve a emissão total; a magnitude medida em uma banda fotométrica corresponde apenas a parte dessa radiação.

## RR Lyrae

RR Lyrae é tanto o nome de uma estrela quanto o nome da classe de variáveis da qual ela é o protótipo. Essas estrelas também pulsam, mas pertencem a populações antigas e costumam apresentar períodos de poucas horas a cerca de um dia. São utilizadas no estudo de populações estelares e de distâncias. Não possuem todas exatamente a mesma luminosidade, e algumas apresentam modulações da amplitude ou da fase ao longo do tempo.

## Eclipsantes

São sistemas com duas estrelas que orbitam um centro de massa comum e cuja orientação permite que uma oculte parte da outra, do nosso ponto de vista. A luz total que recebemos diminui durante o eclipse. Sistemas do tipo Algol costumam apresentar eclipses bem delimitados e brilho relativamente estável entre eles. O eclipse secundário pode ser pouco evidente.

Em uma órbita podem ocorrer duas quedas de brilho. Por isso, o intervalo entre quedas sucessivas nem sempre corresponde ao período orbital. Se os eclipses forem semelhantes, o método pode destacar metade desse período. As definições das classes e subclasses podem ser consultadas no [VSX/AAVSO](https://vsx.aavso.org/index.php?view=about.vartypes).

### Explore a simulação

1. Observe um ciclo de uma pulsante e descreva quais propriedades mudam.
2. Compare a pulsante com o sistema eclipsante. Qual é a causa física da variação em cada caso?
3. Verifique como o sentido da escala de magnitude se relaciona com o brilho.
4. Conte as quedas de brilho por órbita no exemplo eclipsante. Que dificuldade isso pode criar na busca de períodos?

A simulação e sua saída original foram mantidas. Para executar novamente a célula, o arquivo original `simulacao_estrelas_variaveis.html` deve estar na mesma pasta do notebook. A saída já incorporada pode ser visualizada sem repetir essa execução, se o Jupyter a permitir. Se houver bloqueio, marque este notebook como confiável no menu do Jupyter, após reconhecer sua procedência. Você pode iniciar a análise diretamente na seção seguinte; nenhuma variável da simulação é necessária.


In [1]:
from pathlib import Path
from base64 import b64encode
from IPython.display import IFrame, display


arquivo = Path("simulacao_estrelas_variaveis.html")
conteudo = b64encode(arquivo.read_bytes()).decode("ascii")

display(IFrame(
    "data:text/html;base64," + conteudo,
    width="100%",
    height=1400
))

## Importando os dados

### Obtenha e identifique as observações

1. Acesse o [portal da AAVSO](https://www.aavso.org/) e localize a ferramenta de consulta e download de observações ou curvas de luz. Pesquise o identificador exato da estrela escolhida. O acesso pode exigir uma conta.
2. Selecione uma única estrela, uma banda fotométrica e um intervalo que cubra vários ciclos. Comece com um conjunto moderado de observações; muitas décadas de dados podem tornar a primeira busca lenta e misturar épocas com comportamentos diferentes.
3. Exporte os dados em CSV e salve o arquivo na mesma pasta deste notebook, com o nome `dados_estrela.csv`, ou informe seu caminho na configuração abaixo. Preserve a exportação original.
4. Registre nome da estrela, fonte, data da obtenção, banda e seleção temporal. Se o arquivo contiver sinalizadores de qualidade, consulte o significado na documentação da exportação antes de excluir registros.
5. Confira as colunas. O leitor abaixo espera os campos da tabela a seguir; se os nomes forem diferentes, ajuste `MAPA_COLUNAS`. Não renomeie HJD, BJD ou MJD como JD sem compreender e tratar a diferença entre esses sistemas de tempo.

| Coluna | Significado | Unidade ou exemplo |
|---|---|---|
| `JD` | Dia Juliano: contagem contínua de dias; a parte decimal indica uma fração de dia | dias |
| `Magnitude` | Medida do brilho aparente na banda selecionada | mag |
| `Uncertainty` | Incerteza informada da magnitude, usada como desvio-padrão no ajuste | mag |
| `Band` | Banda fotométrica da medição | `V` |
| `Observer Code` | Código de quem realizou a observação | texto |

A banda V corresponde a uma faixa do espectro visível. Não misture bandas diferentes nesta análise: elas podem apresentar magnitudes e amplitudes diferentes. A opção `TODOS` vale apenas para os observadores. Dados de vários observadores podem ter diferenças de calibração; compare subconjuntos se aparecerem deslocamentos sistemáticos.

Não há um CSV de observações anexado a este notebook. Sem ele, a importação interrompe a execução com uma orientação; nenhum conjunto artificial será usado em seu lugar. Para trabalhar sem internet, obtenha previamente o CSV e instale as bibliotecas.

### Prepare o Python

As bibliotecas são conjuntos de ferramentas prontas: Pandas organiza tabelas, NumPy realiza cálculos, Matplotlib produz gráficos, SciPy auxilia a busca numérica e Astropy oferece ferramentas de astronomia. Execute a célula abaixo. Se aparecer `ModuleNotFoundError`, execute em uma nova célula `%pip install numpy pandas matplotlib scipy astropy`; depois reinicie o kernel do Python e retome esta seção.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from urllib.request import urlopen
from urllib.parse import urlparse
import hashlib
import json
import re
import shutil
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from astropy.timeseries import LombScargle
from scipy.signal import find_peaks
from scipy.optimize import minimize_scalar

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Bibliotecas carregadas. Pasta de trabalho:", Path.cwd())


In [ ]:
# CONFIGURAÇÃO DOS DADOS: preencha antes de executar a próxima célula.
ESTRELA = "preencha o identificador da estrela"
CLASSE = "preencha: Cefeida, RR Lyrae ou eclipsante"
ARQUIVO = Path("dados_estrela.csv")
BANDA = "V"
OBSERVADOR = "TODOS"       # Ou um código que realmente exista no arquivo.
JD_INICIO = None            # None mantém todo o intervalo disponível.
JD_FIM = None
FONTE_DADOS = "AAVSO International Database"
DATA_OBTENCAO = ""         # Preencha no formato AAAA-MM-DD.

# Opcional: link HTTPS que devolva diretamente um CSV, fornecido pela fonte
# ou pelo professor. Não use o endereço de uma página de consulta ou de login.
# Se houver arquivo local, ele será usado sem novo download.
URL_CSV = ""

# Ajuste apenas se a exportação usar outros nomes para os mesmos campos.
# Exemplo de sintaxe: MAPA_COLUNAS = {"nome_original": "Magnitude"}
MAPA_COLUNAS = {}
SEPARADOR = None            # Detecção automática; pode ser ",", ";" ou "\t".
DECIMAL = "."

# Exclua flags somente após consultar sua definição na fonte.
COLUNA_QUALIDADE = None     # Exemplo: "Validation Flag", se existir.
FLAGS_EXCLUIR = []          # Lista de códigos documentados, mantida vazia inicialmente.


In [ ]:
def carregar_dados(arquivo):
    """Seleciona uma banda, verifica os campos e registra todas as exclusões."""
    if not arquivo.is_file():
        raise FileNotFoundError(
            f"CSV não encontrado: {arquivo.resolve()}. Salve a exportação da AAVSO "
            "nessa pasta ou corrija ARQUIVO na célula anterior."
        )
    dados = pd.read_csv(arquivo, sep=SEPARADOR, engine="python", comment="#",
                        encoding="utf-8-sig", decimal=DECIMAL)
    dados.columns = dados.columns.str.strip()
    dados = dados.rename(columns=MAPA_COLUNAS)
    obrigatorias = ["JD", "Magnitude", "Uncertainty", "Band"]
    faltantes = [coluna for coluna in obrigatorias if coluna not in dados]
    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}. Encontradas: {list(dados.columns)}")
    if dados.columns.duplicated().any():
        raise ValueError("Há nomes de coluna duplicados; revise MAPA_COLUNAS.")
    print("Bandas disponíveis:", dados["Band"].dropna().astype(str).unique())
    if "Observer Code" in dados:
        print("Observações por observador (até 15 códigos):")
        display(dados["Observer Code"].value_counts().head(15).rename("N"))

    resumo = {"linhas_lidas": len(dados)}
    if not BANDA.strip() or BANDA.strip().upper() == "TODOS":
        raise ValueError("Escolha uma única BANDA, por exemplo V.")
    dados = dados[dados["Band"].astype(str).str.strip().str.upper()
                  == BANDA.strip().upper()].copy()
    resumo["apos_banda"] = len(dados)
    if OBSERVADOR.strip().upper() != "TODOS":
        if "Observer Code" not in dados:
            raise ValueError("O filtro por observador exige a coluna Observer Code.")
        dados = dados[dados["Observer Code"].astype(str).str.strip().str.upper()
                      == OBSERVADOR.strip().upper()].copy()
    resumo["apos_observador"] = len(dados)

    if FLAGS_EXCLUIR and COLUNA_QUALIDADE is None:
        raise ValueError("Defina COLUNA_QUALIDADE para aplicar FLAGS_EXCLUIR.")
    if COLUNA_QUALIDADE is not None:
        if COLUNA_QUALIDADE not in dados:
            raise ValueError(f"Coluna de qualidade ausente: {COLUNA_QUALIDADE}")
        print("Sinalizadores de qualidade antes da seleção:")
        display(dados[COLUNA_QUALIDADE].value_counts(dropna=False))
        flags = dados[COLUNA_QUALIDADE].astype(str).str.strip().str.upper()
        excluir = {str(x).strip().upper() for x in FLAGS_EXCLUIR}
        dados = dados[~flags.isin(excluir)].copy()
    resumo["apos_qualidade"] = len(dados)

    # Limites de detecção não são medições pontuais de magnitude.
    # Alguns arquivos os indicam em uma coluna; outros usam < ou > no valor.
    for coluna in dados.columns:
        chave = re.sub(r"[^a-z]", "", coluna.lower())
        if chave in {"upperlimit", "fainterthan"}:
            flag = dados[coluna].astype(str).str.strip().str.lower()
            limites = flag.isin(["1", "1.0", "true", "t", "yes", "y", "sim", "<", ">"])
            dados = dados[~limites].copy()
    resumo["apos_flags_limite"] = len(dados)

    for coluna in ["JD", "Magnitude", "Uncertainty"]:
        texto = dados[coluna].astype(str).str.strip()
        if DECIMAL != ".":
            texto = texto.str.replace(DECIMAL, ".", regex=False)
        dados[coluna] = pd.to_numeric(texto, errors="coerce")
    valores = dados[["JD", "Magnitude", "Uncertainty"]].to_numpy(float)
    validos = np.isfinite(valores).all(axis=1) & (valores[:, 2] > 0)
    dados = dados[validos].copy()
    resumo["apos_validacao_numerica"] = len(dados)

    if JD_INICIO is not None and JD_FIM is not None and JD_INICIO >= JD_FIM:
        raise ValueError("JD_INICIO deve ser menor que JD_FIM.")
    if JD_INICIO is not None:
        dados = dados[dados["JD"] >= JD_INICIO].copy()
    if JD_FIM is not None:
        dados = dados[dados["JD"] <= JD_FIM].copy()
    dados = dados.sort_values("JD").reset_index(drop=True)
    resumo["apos_intervalo"] = len(dados)
    display(pd.DataFrame(list(resumo.items()), columns=["Etapa", "N de registros"]))
    if len(dados) < 20 or dados["JD"].nunique() < 10:
        raise ValueError("A atividade exige ao menos 20 medidas válidas em 10 instantes "
                         "distintos. Revise os filtros ou obtenha mais observações. "
                         "Esses mínimos operacionais não garantem um período confiável.")
    if np.ptp(dados["JD"]) <= 0 or np.ptp(dados["Magnitude"]) <= 0:
        raise ValueError("É necessário haver variação nos tempos e nas magnitudes.")
    return dados, resumo


# Download opcional de uma exportação CSV já disponibilizada.
# Esta opção não implementa uma API específica da AAVSO.
if not ARQUIVO.exists() and URL_CSV.strip():
    if urlparse(URL_CSV).scheme != "https":
        raise ValueError("URL_CSV deve ser um endereço HTTPS direto para o CSV.")
    with urlopen(URL_CSV, timeout=30) as resposta:
        conteudo_csv = resposta.read(25_000_001)
    if len(conteudo_csv) > 25_000_000:
        raise ValueError("Exportação maior que 25 MB; baixe um intervalo menor para a atividade.")
    inicio = conteudo_csv[:1000].lower()
    if b"<html" in inicio or b"<!doctype html" in inicio:
        raise ValueError("O endereço devolveu uma página HTML, não um arquivo CSV.")
    ARQUIVO.parent.mkdir(parents=True, exist_ok=True)
    ARQUIVO.write_bytes(conteudo_csv)
    if not DATA_OBTENCAO:
        DATA_OBTENCAO = datetime.now(timezone.utc).date().isoformat()

dados, resumo_limpeza = carregar_dados(ARQUIVO)
tempo = dados["JD"].to_numpy(float)
magnitude = dados["Magnitude"].to_numpy(float)
erro = dados["Uncertainty"].to_numpy(float)
tempo_referencia = float(tempo.min())
tempo_relativo = tempo - tempo_referencia
duracao = float(np.ptp(tempo))
sha256_dados = hashlib.sha256(ARQUIVO.read_bytes()).hexdigest()
print(f"{len(dados)} observações selecionadas; duração total: {duracao:.3f} dias.")
display(dados.head())


O código exclui campos não numéricos, limites de detecção identificados e registros sem incerteza positiva. Uma incerteza ausente não prova que a observação esteja errada: apenas impede seu uso neste ajuste ponderado. Não atribua um valor arbitrário para fazê-la passar pelo filtro.

Não aplicamos corte automático pela distância à magnitude mediana. Um ponto extremo pode ser um eclipse ou parte real de uma pulsação. Examine os dados e a qualidade antes de decidir por qualquer exclusão. A tabela de contagens permite acompanhar o efeito dos critérios aplicados.

O download opcional automatiza a obtenção de um CSV por um link direto. A integração específica com a API da AAVSO, prevista como objetivo do projeto, depende da interface vigente e não é implementada neste guia. Guardar a exportação original permite repetir a análise mesmo que o serviço mude.

Atividade: quantas medidas foram mantidas? Qual critério retirou mais registros? A série selecionada contém uma única banda e cobre um intervalo adequado para a classe escolhida? Registre os valores, não apenas “sim” ou “não”.


## Curva de luz

A curva de luz representa como o brilho observado varia com o tempo. Cada ponto é uma observação; a barra vertical representa sua incerteza. As lacunas mostram intervalos sem medidas. Não ligamos os pontos, pois uma linha poderia sugerir que conhecemos o que ocorreu durante essas lacunas.

Na escala de magnitudes, números menores correspondem a maior brilho. Uma observação de magnitude 8 indica mais luz recebida que uma de magnitude 9, na mesma banda. A relação entre magnitudes e fluxos recebidos é

$$m_2-m_1=-2{,}5\log_{10}\!\left(\frac{F_2}{F_1}\right).$$

Uma diminuição de uma magnitude corresponde a um aumento do fluxo por um fator de aproximadamente 2,512. “Magnitude mínima” e “brilho mínimo” têm sentidos opostos. Invertemos o eixo vertical para que maior brilho apareça mais acima.

Para facilitar a leitura, o eixo horizontal mostra os dias decorridos desde a primeira observação; a data de referência em JD aparece no rótulo. Subtrair essa referência não altera os intervalos nem o período.


In [ ]:
def plotar_curva_luz(tempo_relativo, magnitude, erro):
    figura, eixo = plt.subplots(figsize=(10, 4.5))
    eixo.errorbar(tempo_relativo, magnitude, yerr=erro, fmt=".", markersize=4,
                 alpha=0.6, elinewidth=0.5, capsize=0)
    eixo.invert_yaxis()
    eixo.set(xlabel=f"Tempo desde JD {tempo_referencia:.5f} (dias)",
             ylabel=f"Magnitude na banda {BANDA}", title=f"Curva de luz: {ESTRELA}")
    figura.tight_layout()
    return figura

figura_curva = plotar_curva_luz(tempo_relativo, magnitude, erro)
plt.show()
amplitude_observada = float(np.ptp(magnitude))
print(f"Amplitude pico a pico da amostra: {amplitude_observada:.3f} mag.")
print("Esse valor é máximo(magnitude) - mínimo(magnitude); pode ser afetado por "
      "pontos discrepantes e pela falta de observações nos extremos do ciclo.")


Atividade: identifique um intervalo com maior densidade de observações e uma lacuna. A variação de magnitude parece maior que as incertezas? É possível estimar um tempo de repetição apenas com esse gráfico? Justifique com características da curva.


## Busca de períodos

O período $P$ é a duração de um ciclo. A frequência $f$ indica quantas vezes um ciclo se repete no tempo.

$$f=\frac{1}{P}.$$

Se $P$ está em dias, $f$ está em ciclos por dia. Uma estrela com período de dois dias tem frequência de 0,5 ciclo por dia.

Noites nubladas, luz do dia, posição da estrela no céu e disponibilidade dos observadores produzem amostragem irregular. A FFT convencional exige uma grade temporal regular para seu uso direto; isso não significa que toda análise de Fourier seja impossível com dados irregulares.

Lomb–Scargle compara ajustes senoidais em muitas frequências. Nesta implementação, considera as incertezas das medidas e ajusta também um nível médio. Você não precisa deduzir o método para interpretar o resultado: buscamos os ritmos que descrevem melhor os dados. [Referência: VanderPlas, 2018](https://arxiv.org/abs/1703.09824).

Escolha um intervalo de busca amplo o suficiente para investigar sua hipótese, sem usar o período exato do catálogo como resposta antecipada. Os valores abaixo são uma configuração inicial para a atividade, não limites universais das classes. Para RR Lyrae, por exemplo, uma primeira exploração de 0,2 a 2 dias é mais econômica. Para Cefeidas, pode-se começar com 1 a 100 dias. Em eclipsantes, escolha os limites conforme o objeto e a duração da série.

Como regra prática didática, a busca abaixo exige espaço para ao menos dois ciclos dentro da duração observada e informa qualquer redução do limite máximo. Isso não garante que as fases tenham sido bem observadas. A grade usa dez amostras por largura típica de pico; uma grade mais fina não cria informação nova.


In [ ]:
# CONFIGURAÇÃO DA BUSCA: períodos em dias.
PERIODO_MINIMO = 0.2
PERIODO_MAXIMO = 100.0
AMOSTRAS_POR_PICO = 10
MAX_FREQUENCIAS = 1_000_000  # Limite prático para computadores escolares.

if not (np.isfinite(PERIODO_MINIMO) and np.isfinite(PERIODO_MAXIMO)
        and 0 < PERIODO_MINIMO < PERIODO_MAXIMO):
    raise ValueError("Use 0 < PERIODO_MINIMO < PERIODO_MAXIMO, com valores finitos.")
if AMOSTRAS_POR_PICO < 5:
    raise ValueError("Nesta atividade, use ao menos 5 amostras por pico.")
periodo_maximo_efetivo = min(PERIODO_MAXIMO, duracao / 2)
if periodo_maximo_efetivo <= PERIODO_MINIMO:
    raise ValueError("Série curta demais para o intervalo escolhido. Revise os dados ou os limites.")
if periodo_maximo_efetivo < PERIODO_MAXIMO:
    print(f"Limite máximo reduzido para {periodo_maximo_efetivo:.5g} dias "
          "pela duração das observações.")
frequencia_minima = 1 / periodo_maximo_efetivo
frequencia_maxima = 1 / PERIODO_MINIMO
estimativa_grade = int(np.ceil((frequencia_maxima - frequencia_minima)
                              * duracao * AMOSTRAS_POR_PICO)) + 1
if estimativa_grade > MAX_FREQUENCIAS:
    raise ValueError(f"A busca exigiria cerca de {estimativa_grade:,} frequências. "
                     "Estreite o intervalo de períodos ou selecione um intervalo temporal "
                     "menor e reexecute desde a importação.")

ls = LombScargle(tempo_relativo, magnitude, dy=erro, fit_mean=True,
                center_data=True, nterms=1, normalization="standard")
frequencias = ls.autofrequency(minimum_frequency=frequencia_minima,
                              maximum_frequency=frequencia_maxima,
                              samples_per_peak=AMOSTRAS_POR_PICO)
frequencias = frequencias[(frequencias >= frequencia_minima)
                          & (frequencias <= frequencia_maxima)]
if len(frequencias) < 3:
    raise ValueError("Intervalo de busca estreito demais; amplie os limites de período.")
potencia = ls.power(frequencias)
if not np.isfinite(potencia).all():
    raise ValueError("Há potências não finitas. Revise a série e os limites da busca.")
indice_maximo = int(np.argmax(potencia))
periodo_pico_ls = float(1 / frequencias[indice_maximo])
print(f"Busca: {PERIODO_MINIMO:g} a {periodo_maximo_efetivo:g} dias; "
      f"{len(frequencias):,} frequências testadas.")


## Periodograma

O periodograma mostra a qualidade relativa dos ajustes em cada período testado. A potência é uma medida estatística sem unidade nesta normalização, não a potência luminosa da estrela. Um pico alto identifica um candidato que merece investigação.

As linhas de probabilidade de falso alarme (FAP) ajudam a avaliar a altura dos picos. Por exemplo, FAP de 1% significa que, sob a hipótese de brilho constante com ruído gaussiano independente, seria raro obter um pico tão alto ou maior na busca. Não significa “99% de chance de o período estar correto”. Erros sistemáticos e a distribuição dos instantes de observação exigem análise adicional. [Documentação do Astropy sobre FAP](https://docs.astropy.org/en/stable/timeseries/lombscargle.html#peak-significance-and-false-alarm-probabilities).

O eixo horizontal é logarítmico: espaçamentos iguais correspondem a razões iguais entre períodos. A linha vertical marca o maior pico da grade; a seleção será examinada nas próximas etapas.


In [ ]:
figura_periodograma, eixo = plt.subplots(figsize=(10, 4.5))
periodos_grade = 1 / frequencias
ordem = np.argsort(periodos_grade)
eixo.plot(periodos_grade[ordem], potencia[ordem], lw=0.8, color="tab:blue")
parametros_fap = dict(method="baluev", minimum_frequency=frequencia_minima,
                      maximum_frequency=frequencia_maxima,
                      samples_per_peak=AMOSTRAS_POR_PICO)
fap_pico = None
try:
    fap_pico = float(ls.false_alarm_probability(potencia[indice_maximo], **parametros_fap))
    if not np.isfinite(fap_pico):
        fap_pico = None
        raise ValueError("FAP não finita")
    probabilidades = [0.10, 0.05, 0.01]
    niveis = ls.false_alarm_level(probabilidades, **parametros_fap)
    for probabilidade, nivel, cor in zip(probabilidades, niveis,
                                         ["#b98400", "#d65f00", "#b00020"]):
        if np.isfinite(nivel):
            eixo.axhline(nivel, color=cor, ls="--", lw=1,
                         label=f"FAP {probabilidade:.0%}")
    print(f"FAP estimada para o maior pico da grade: {fap_pico:.3g}.")
    print("Um valor 0 pode representar um número abaixo da precisão numérica.")
except (ValueError, FloatingPointError, OverflowError) as exc:
    print("Não foi possível obter todos os níveis de FAP:", exc)

eixo.axvline(periodo_pico_ls, color="black", ls=":", label="Maior pico da grade")
eixo.set(xscale="log", xlabel="Período testado (dias)",
         ylabel="Potência Lomb–Scargle", title=f"Periodograma: {ESTRELA}")
eixo.legend(fontsize=9)
figura_periodograma.tight_layout()
plt.show()


Atividade: há um pico isolado ou vários picos de altura semelhante? O maior pico ultrapassa algum dos níveis de FAP? Explique o que essa informação permite concluir e o que ainda precisa ser verificado.


## Encontrando o período

O maior pico é um ponto de partida. A amostragem pode produzir aliases: períodos concorrentes associados à forma como os dados foram obtidos. Além disso, curvas que não são senoidais podem destacar harmônicos, como $P/2$. Um pico próximo de um dia não deve ser descartado apenas por essa proximidade.

O código lista até cinco picos separados por aproximadamente $1/T$ em frequência, sendo $T$ a duração da série, e refina numericamente seus arredores. Essa separação evita repetir o mesmo pico na tabela; não identifica nem elimina aliases automaticamente. O refinamento melhora a localização numérica, mas não fornece a incerteza do período. [Discussão sobre amostragem e periodicidade: VanderPlas, 2018](https://arxiv.org/abs/1703.09824).

Você investigará o candidato principal, sua metade e seu dobro no diagrama de fase. Registre o resultado antes de consultar o período do catálogo.


In [ ]:
resolucao_frequencia = 1 / duracao
passo_frequencia = float(np.median(np.diff(frequencias)))
distancia_indices = max(1, int(np.ceil(resolucao_frequencia / passo_frequencia)))
indices, _ = find_peaks(potencia, distance=distancia_indices)
bordas = []
if potencia[0] >= potencia[1]:
    bordas.append(0)
if potencia[-1] >= potencia[-2]:
    bordas.append(len(potencia) - 1)
indices = np.unique(np.r_[indices, indice_maximo, bordas]).astype(int)
indices = indices[np.argsort(potencia[indices])[::-1]][:5]

def refinar_pico(indice):
    """Refina dentro dos vizinhos imediatos da grade, sem saltar entre picos."""
    inferior = float(frequencias[max(0, indice - 1)])
    superior = float(frequencias[min(len(frequencias) - 1, indice + 1)])
    central = float(frequencias[indice])
    candidatos_locais = [inferior, central, superior]
    if superior > inferior:
        resultado = minimize_scalar(
            lambda f: -float(ls.power(f, method="cython")),
            bounds=(inferior, superior), method="bounded",
            options={"xatol": max(1e-12, (superior - inferior) * 1e-5)}
        )
        if resultado.success and np.isfinite(resultado.fun):
            candidatos_locais.append(float(resultado.x))
    potencias_locais = [float(ls.power(f, method="cython")) for f in candidatos_locais]
    melhor = int(np.argmax(potencias_locais))
    return candidatos_locais[melhor], potencias_locais[melhor]

linhas = []
for indice in indices:
    frequencia, altura = refinar_pico(indice)
    linhas.append({"Periodo_dias": 1 / frequencia,
                   "Frequencia_ciclos_dia": frequencia, "Potencia": altura})
tabela_candidatos = pd.DataFrame(linhas).sort_values("Potencia", ascending=False).reset_index(drop=True)
tabela_candidatos.index = tabela_candidatos.index + 1
tabela_candidatos.index.name = "Candidato"
display(tabela_candidatos.round(7))
periodo_candidato = float(tabela_candidatos.iloc[0]["Periodo_dias"])
print(f"Candidato inicial: {periodo_candidato:.7g} dias = {24 * periodo_candidato:.7g} horas.")
if indice_maximo in [0, len(frequencias) - 1]:
    print("O maior pico está na borda da busca. Amplie os limites antes de interpretar o período.")


## Diagrama de fase

Para comparar ciclos observados em datas diferentes, calculamos a posição de cada medida dentro de um ciclo:

$$\phi=\left(\frac{t-t_0}{P}\right)\bmod 1.$$

O símbolo “mod 1” indica a parte fracionária: depois de contar os ciclos completos, guardamos apenas a fração restante. Uma medida obtida 2,25 períodos após $t_0$ tem fase 0,25, assim como uma medida obtida 5,25 períodos depois. A fase não tem unidade.

Aqui $t_0$ é a primeira data observada. Portanto, fase zero não significa necessariamente brilho máximo nem centro de eclipse. Usamos a mesma referência ao comparar os períodos.

Se o período for adequado e a variação for suficientemente estável, os pontos tendem a delinear uma curva. A dispersão não desaparece: pode resultar das incertezas, de calibrações diferentes, de modulações reais ou de um período inadequado.

Execute a comparação. Depois, se necessário, preencha `PERIODO_ESCOLHIDO` com outro candidato em dias e escreva sua justificativa. Reexecute esta célula e todas as seguintes. O período adotado continua sendo uma hipótese de trabalho; não basta escolher o desenho visualmente mais agradável.


In [ ]:
# None adota o maior pico refinado. Para investigar outro, informe seu período.
PERIODO_ESCOLHIDO = None
JUSTIFICATIVA_PERIODO = ""  # Registre quais evidências sustentam sua escolha.
N_BINS = 25                # Quantidade de intervalos de fase para a visualização.

periodo_adotado = (periodo_candidato if PERIODO_ESCOLHIDO is None
                  else float(PERIODO_ESCOLHIDO))
if not np.isfinite(periodo_adotado) or periodo_adotado <= 0:
    raise ValueError("O período escolhido deve ser positivo e finito.")
if not (PERIODO_MINIMO <= periodo_adotado <= periodo_maximo_efetivo):
    raise ValueError("O período escolhido está fora da busca. Amplie os limites "
                     "e reexecute desde Busca de períodos.")
if not isinstance(N_BINS, int) or N_BINS < 2:
    raise ValueError("N_BINS deve ser um inteiro maior ou igual a 2.")

def calcular_fase(periodo):
    return np.remainder(tempo_relativo / periodo, 1.0)

def medias_por_fase(fase, n_bins):
    """Médias ponderadas para visualização; o ajuste usa todos os pontos originais."""
    indices_bins = np.minimum((fase * n_bins).astype(int), n_bins - 1)
    linhas = []
    for i in range(n_bins):
        mascara = indices_bins == i
        if np.count_nonzero(mascara) >= 2:
            pesos = 1 / erro[mascara] ** 2
            linhas.append([np.average(fase[mascara], weights=pesos),
                           np.average(magnitude[mascara], weights=pesos),
                           np.sqrt(1 / pesos.sum())])
    return np.asarray(linhas).reshape(-1, 3)

figura_comparacao, eixos = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for eixo, fator, rotulo in zip(eixos, [0.5, 1, 2], ["P/2", "P adotado", "2P"]):
    periodo_teste = fator * periodo_adotado
    eixo.errorbar(calcular_fase(periodo_teste), magnitude, yerr=erro,
                  fmt=".", ms=3, alpha=0.35, elinewidth=0.4)
    fora = not (PERIODO_MINIMO <= periodo_teste <= periodo_maximo_efetivo)
    eixo.set(xlabel="Fase", xlim=(0, 1),
             title=f"{rotulo}: {periodo_teste:.6g} d" + ("\nfora da busca" if fora else ""))
eixos[0].set_ylabel(f"Magnitude na banda {BANDA}")
eixos[0].invert_yaxis()
figura_comparacao.tight_layout()
plt.show()
fase = calcular_fase(periodo_adotado)
medias_fase = medias_por_fase(fase, N_BINS)
ocupacao = np.histogram(fase, bins=N_BINS, range=(0, 1))[0]
print(f"Cobertura: {np.count_nonzero(ocupacao)}/{N_BINS} intervalos de fase com observações.")


Atividade: compare a dispersão, as lacunas de fase e o número de máximos ou mínimos em cada painel. Em uma eclipsante, dois eclipses de profundidades diferentes se sobrepõem quando você utiliza um período inadequado? Em uma pulsante, dobrar o período repete a mesma forma duas vezes? Se os eclipses forem praticamente iguais, a fotometria disponível pode não resolver sozinha a ambiguidade entre período fotométrico e orbital.

Os pontos médios por intervalo de fase serão mostrados no gráfico seguinte. Agrupar os pontos ajuda a visualizar a tendência, mas não corrige calibrações nem elimina variações físicas. Intervalos largos podem esconder eclipses estreitos. As barras das médias representam apenas a incerteza formal propagada das medidas, supondo independência; não incluem toda a dispersão dentro do intervalo.


## Ajuste de Fourier

Uma soma de senos e cossenos permite aproximar a forma de uma curva periódica:

$$m(\phi)=a_0+\sum_{k=1}^{K}\left[a_k\sin(2\pi k\phi)+b_k\cos(2\pi k\phi)\right].$$

$a_0$ representa o nível médio do modelo. O inteiro $k$ identifica os harmônicos, que têm frequências múltiplas da frequência fundamental; $K$ indica quantos deles são incluídos. Com um harmônico, o modelo é senoidal. Mais harmônicos permitem formas assimétricas, mas também podem ajustar ruído ou oscilar artificialmente nas lacunas.

O ajuste usa todas as observações selecionadas, com pesos $1/\sigma_i^2$: medidas com menor incerteza têm maior peso. As médias de fase aparecem apenas como auxílio visual. A linha ajustada é uma descrição matemática aproximada dos dados, não um modelo completo do interior da estrela ou da geometria orbital.

Mantemos fixo o período escolhido na etapa anterior e comparamos modelos com um a quatro harmônicos pelo BIC, critério que penaliza complexidade excessiva. Para as mesmas observações e incertezas gaussianas fixas, usamos $\mathrm{BIC}=\chi^2+q\ln N$, omitindo uma constante comum; $q=1+2K$ é o número de coeficientes e $N$ é o número de medidas. Menor BIC favorece um modelo entre os testados; não comprova a classe da estrela nem resolve automaticamente aliases.

Os resíduos são as diferenças “observação menos modelo”. Observe se apresentam estrutura, principalmente perto dos eclipses. O RMS ponderado resume seu tamanho em magnitudes. O $\chi^2$ reduzido é apenas um diagnóstico aproximado aqui: o período também foi escolhido a partir dos dados e isso não entra na contagem de coeficientes do ajuste de forma.


In [ ]:
N_TERMOS_MAXIMO = 4  # Aprofundamento: compare valores de 1 a 6.
if not isinstance(N_TERMOS_MAXIMO, int) or not 1 <= N_TERMOS_MAXIMO <= 6:
    raise ValueError("Nesta atividade, escolha de 1 a 6 harmônicos.")

def matriz_fourier(fase_modelo, n_termos):
    colunas = [np.ones_like(fase_modelo)]
    for k in range(1, n_termos + 1):
        angulo = 2 * np.pi * k * fase_modelo
        colunas.extend([np.sin(angulo), np.cos(angulo)])
    return np.column_stack(colunas)

def avaliar_fourier(n_termos):
    matriz = matriz_fourier(fase, n_termos)
    q = matriz.shape[1]
    if len(magnitude) <= q:
        raise ValueError("Poucas medidas para essa quantidade de coeficientes.")
    coeficientes, _, posto, singulares = np.linalg.lstsq(
        matriz / erro[:, None], magnitude / erro, rcond=None)
    if posto < q or singulares[-1] <= 0:
        raise ValueError("Fases insuficientes para determinar todos os coeficientes.")
    condicao = float(singulares[0] / singulares[-1])
    if condicao > 1e10:
        raise ValueError("Ajuste instável: reduza os harmônicos ou melhore a cobertura de fase.")
    residuos = magnitude - matriz @ coeficientes
    chi2 = float(np.sum((residuos / erro) ** 2))
    return {"K": n_termos, "BIC": chi2 + q * np.log(len(magnitude)),
            "RMS_mag": float(np.sqrt(np.average(residuos ** 2, weights=1 / erro ** 2))),
            "Chi2_reduzido": chi2 / (len(magnitude) - q),
            "coeficientes": coeficientes, "residuos": residuos}

ajustes = []
for k in range(1, N_TERMOS_MAXIMO + 1):
    try:
        ajustes.append(avaliar_fourier(k))
    except (ValueError, np.linalg.LinAlgError) as exc:
        print(f"Modelo com {k} harmônicos não utilizado:", exc)
if not ajustes:
    raise ValueError("Nenhum ajuste estável; reveja a seleção dos dados e o período.")
melhor_ajuste = min(ajustes, key=lambda ajuste: ajuste["BIC"])
tabela_ajustes = pd.DataFrame([{chave: a[chave] for chave in
                               ["K", "BIC", "RMS_mag", "Chi2_reduzido"]} for a in ajustes])
display(tabela_ajustes.round(4))

fase_modelo = np.linspace(0, 1, 2000)
magnitude_modelo = (matriz_fourier(fase_modelo, melhor_ajuste["K"])
                    @ melhor_ajuste["coeficientes"])
amplitude_modelo = float(np.ptp(magnitude_modelo))
figura_ajuste, (eixo, eixo_residuos) = plt.subplots(
    2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
eixo.errorbar(fase, magnitude, yerr=erro, fmt=".", ms=4, alpha=0.3,
              elinewidth=0.4, label="Observações")
if len(medias_fase):
    eixo.errorbar(medias_fase[:, 0], medias_fase[:, 1], yerr=medias_fase[:, 2],
                  fmt="o", ms=4, color="tab:red", capsize=2, label="Médias por intervalo de fase")
eixo.plot(fase_modelo, magnitude_modelo, color="tab:green", lw=2,
          label=f"Fourier: {melhor_ajuste['K']} harmônicos")
eixo.invert_yaxis()
eixo.set(ylabel=f"Magnitude na banda {BANDA}",
         title=f"{ESTRELA} · P adotado = {periodo_adotado:.7g} dias")
eixo.legend(fontsize=9)
eixo_residuos.errorbar(fase, melhor_ajuste["residuos"], yerr=erro,
                       fmt=".", ms=3, alpha=0.4, elinewidth=0.4)
eixo_residuos.axhline(0, color="black", lw=1)
eixo_residuos.set(xlabel="Fase", ylabel="Resíduo (mag)", xlim=(0, 1))
figura_ajuste.tight_layout()
plt.show()
print(f"Amplitude pico a pico do modelo: {amplitude_modelo:.3f} mag.")
print("Em fases pouco observadas, os extremos do modelo podem ser extrapolações frágeis.")


Atividade: o modelo acompanha as medidas nas regiões com boa cobertura? Há oscilações da linha onde quase não existem dados? Compare a amplitude da amostra com a do modelo e explique uma possível diferença. Aumentar o número de harmônicos melhorou a descrição de uma característica observada ou apenas tornou a linha mais irregular?

### Valide os resultados com um catálogo

Esta comparação integra o objetivo do projeto de confrontar a análise com fontes reconhecidas. Faça-a depois de registrar sua estimativa inicial.

1. Pesquise o identificador no [VSX](https://vsx.aavso.org/). Confirme que se trata do mesmo objeto; o [SIMBAD](https://simbad.cds.unistra.fr/simbad/) pode auxiliar na identificação, nas coordenadas e nos nomes alternativos.
2. Anote o período de referência, a classificação, as magnitudes nos extremos, a banda, a página específica consultada e a data. Leia eventuais marcas de incerteza do catálogo.
3. Compare períodos expressos na mesma unidade e verifique se a referência indica período orbital, período de pulsação ou outro ciclo.
4. Compare amplitudes somente na mesma banda. A diferença entre os extremos catalogados não é necessariamente igual à amplitude de uma série curta ou do modelo ajustado.

Calcularemos a diferença relativa percentual:

$$\Delta P_{\%}=100\frac{|P_{\mathrm{adotado}}-P_{\mathrm{ref}}|}{P_{\mathrm{ref}}}.$$

Uma diferença pequena indica concordância numérica, mas não é uma estimativa de incerteza nem uma validação independente de todos os procedimentos. Se a razão entre os períodos estiver próxima de 0,5 ou 2, investigue a hipótese de harmônicos e o significado do período do catálogo. Diferenças também podem resultar de amostragem, calibração ou mudanças físicas ao longo dos anos. Nenhum valor de referência é preenchido automaticamente.


In [ ]:
# PREENCHA SOMENTE APÓS CONSULTAR O MESMO OBJETO NO CATÁLOGO.
PERIODO_REFERENCIA = None        # Dias; use um número, sem aspas.
TIPO_CATALOGO = ""
MAG_MAIS_BRILHANTE_REF = None    # Menor valor de magnitude.
MAG_MENOS_BRILHANTE_REF = None   # Maior valor de magnitude.
BANDA_REFERENCIA = ""
URL_REFERENCIA = ""
DATA_CONSULTA = ""              # AAAA-MM-DD.

amplitude_referencia = None
diferenca_periodo_pct = None
razao_periodos = None
if PERIODO_REFERENCIA is not None:
    if not np.isfinite(PERIODO_REFERENCIA) or PERIODO_REFERENCIA <= 0:
        raise ValueError("O período de referência deve ser positivo e finito.")
    diferenca_periodo_pct = 100 * abs(periodo_adotado - PERIODO_REFERENCIA) / PERIODO_REFERENCIA
    razao_periodos = periodo_adotado / PERIODO_REFERENCIA
    print(f"Diferença relativa: {diferenca_periodo_pct:.5g}%; razão P/P_ref: {razao_periodos:.7g}.")
else:
    print("Comparação do período pendente: preencha o valor de referência e sua fonte.")

if MAG_MAIS_BRILHANTE_REF is not None and MAG_MENOS_BRILHANTE_REF is not None:
    if (not np.isfinite([MAG_MAIS_BRILHANTE_REF, MAG_MENOS_BRILHANTE_REF]).all()
            or MAG_MENOS_BRILHANTE_REF < MAG_MAIS_BRILHANTE_REF):
        raise ValueError("Revise os extremos de magnitude do catálogo.")
    if BANDA_REFERENCIA.strip().upper() == BANDA.strip().upper():
        amplitude_referencia = MAG_MENOS_BRILHANTE_REF - MAG_MAIS_BRILHANTE_REF
    else:
        print("A amplitude de referência não será comparada: as bandas são diferentes ou não informadas.")

tabela_resultado = pd.DataFrame([{
    "Estrela": ESTRELA, "Classe_investigada": CLASSE, "Banda": BANDA,
    "N": len(dados), "Duracao_dias": duracao, "P_adotado_dias": periodo_adotado,
    "P_referencia_dias": PERIODO_REFERENCIA, "Diferenca_P_pct": diferenca_periodo_pct,
    "Amplitude_amostra_mag": amplitude_observada, "Amplitude_modelo_mag": amplitude_modelo,
    "Amplitude_referencia_mag": amplitude_referencia, "Tipo_catalogo": TIPO_CATALOGO,
    "Harmonicos": melhor_ajuste["K"], "RMS_mag": melhor_ajuste["RMS_mag"],
    "URL_referencia": URL_REFERENCIA, "Data_consulta": DATA_CONSULTA
}])
display(tabela_resultado.T.rename(columns={0: "Resultado"}))


## Indicações dos autores

Recomendamos estudar noções de Python em paralelo às atividades: variáveis, listas, funções, tabelas e leitura de gráficos. O domínio completo da programação não é um pré-requisito para começar. Primeiro reproduza uma análise; depois altere um parâmetro por vez e explique a mudança observada.

### Para consultar durante a atividade

- [AAVSO](https://www.aavso.org/): origem das observações e acesso às ferramentas de dados.
- [VSX: tipos de estrelas variáveis](https://vsx.aavso.org/index.php?view=about.vartypes): significado das classificações.
- [VSX](https://vsx.aavso.org/): consulta aos objetos e seus parâmetros de referência.
- [SIMBAD](https://simbad.cds.unistra.fr/simbad/): identificação de objetos e bibliografia.
- [Astropy: Lomb–Scargle](https://docs.astropy.org/en/stable/timeseries/lombscargle.html): documentação do método e de suas limitações.
- [VanderPlas (2018), Understanding the Lomb–Scargle Periodogram](https://arxiv.org/abs/1703.09824): aprofundamento sobre periodicidade e amostragem.

### Se ocorrer um problema

| Situação | O que verificar |
|---|---|
| `FileNotFoundError` na importação | Nome e caminho do CSV; a célula de bibliotecas mostra a pasta de trabalho. |
| `ModuleNotFoundError` | Instalação da biblioteca no kernel em uso, conforme “Prepare o Python”. |
| `NameError` | Execute as células de análise na ordem, começando em “Importando os dados”. |
| Nenhuma medida ou poucas medidas após os filtros | Banda, observador, datas, incertezas e colunas; confira a tabela de contagens. |
| Período diferente do catálogo | Identidade do objeto, unidade, banda, cobertura de fase, aliases e possibilidade de P/2 ou 2P. |
| Curva de fase muito dispersa | Experimente os outros candidatos e compare observadores e intervalos temporais; investigue também variação real. |

Para uso em aula, o professor pode fornecer previamente um CSV por classe e dividir a atividade em leitura conceitual, investigação dos dados e comparação dos resultados. Para compartilhar o trabalho, inclua os arquivos de dados com sua procedência e respeite os termos de uso da fonte. A disponibilização pública e a licença do guia devem ser definidas pelos autores.


## Conclusão

Uma análise de estrelas variáveis reúne observação, cálculo e interpretação física. O periodograma sugere ritmos; o diagrama de fase ajuda a examiná-los; o ajuste descreve a forma da curva; a comparação com o catálogo situa o resultado. Nenhuma dessas etapas, isoladamente, torna o período definitivo.

### Registre a investigação

Produza uma síntese curta com:

1. A estrela e sua classe, a origem dos dados, a banda, o intervalo e os critérios de seleção.
2. A curva de luz, o periodograma, o diagrama de fase e o período adotado, com unidade e justificativa.
3. A comparação com o catálogo, citando a página consultada, e a discussão das diferenças.
4. Uma limitação concreta dos dados ou do método e uma ação que poderia melhorar a análise.

Repita o procedimento para uma Cefeida, uma RR Lyrae e uma eclipsante, conforme a proposta do projeto. Reúna as três linhas de resultados e compare: quais características do período e da forma da curva ajudam a distinguir os mecanismos físicos? Quais características se sobrepõem e impedem uma classificação apenas pelo gráfico?

### Guarde arquivos que permitam reproduzir os resultados

A próxima célula é opcional. Depois de concluir a análise e preencher a identificação dos dados, altere `SALVAR_RESULTADOS` para `True`. Ela cria uma pasta nova em `resultados/` com a exportação original, a seleção analisada, tabelas, gráficos e os parâmetros da execução. A comparação permanece indicada como pendente enquanto a referência não for preenchida. As casas decimais dos cálculos não representam uma incerteza determinada.

Salve também o próprio notebook pelo menu do Jupyter, mantendo suas respostas. Para retomar uma análise, conserve os dados e use novamente as configurações registradas. Para uma distribuição em HTML ou PDF, exporte uma cópia pelo Jupyter; o notebook continua sendo a versão executável, e o PDF não preserva a interatividade da simulação.


In [ ]:
SALVAR_RESULTADOS = False

if SALVAR_RESULTADOS:
    if (not ESTRELA.strip() or ESTRELA.startswith("preencha")
            or not CLASSE.strip() or CLASSE.startswith("preencha") or not DATA_OBTENCAO):
        raise ValueError("Preencha ESTRELA, CLASSE e DATA_OBTENCAO na configuração dos dados.")
    identificador = re.sub(r"[^A-Za-z0-9_-]+", "_", ESTRELA).strip("_") or "estrela"
    instante = datetime.now(timezone.utc)
    pasta_saida = Path("resultados") / f"{identificador}_{instante:%Y%m%dT%H%M%S%fZ}"
    pasta_saida.mkdir(parents=True, exist_ok=False)
    shutil.copyfile(ARQUIVO, pasta_saida / "observacoes_originais.csv")
    dados.to_csv(pasta_saida / "dados_selecionados.csv", index=False)
    tabela_candidatos.to_csv(pasta_saida / "periodos_candidatos.csv")
    tabela_ajustes.to_csv(pasta_saida / "comparacao_fourier.csv", index=False)
    tabela_resultado.to_csv(pasta_saida / "resultado.csv", index=False)
    for nome, figura in [("curva_luz", figura_curva), ("periodograma", figura_periodograma),
                          ("comparacao_fases", figura_comparacao), ("ajuste_fourier", figura_ajuste)]:
        figura.savefig(pasta_saida / f"{nome}.png", dpi=160, bbox_inches="tight")
    registro = {
        "estrela": ESTRELA, "classe_investigada": CLASSE,
        "fonte_dados": FONTE_DADOS, "url_csv": URL_CSV,
        "data_obtencao": DATA_OBTENCAO, "execucao_utc": instante.isoformat(),
        "arquivo_original": ARQUIVO.name, "sha256_original": sha256_dados,
        "banda": BANDA, "observador": OBSERVADOR,
        "jd_inicio_filtro": JD_INICIO, "jd_fim_filtro": JD_FIM,
        "mapa_colunas": MAPA_COLUNAS, "separador": SEPARADOR, "decimal": DECIMAL,
        "coluna_qualidade": COLUNA_QUALIDADE, "flags_excluidas": FLAGS_EXCLUIR,
        "contagens_selecao": resumo_limpeza, "t0_jd": tempo_referencia,
        "periodo_minimo": PERIODO_MINIMO, "periodo_maximo_solicitado": PERIODO_MAXIMO,
        "periodo_maximo_efetivo": periodo_maximo_efetivo,
        "amostras_por_pico": AMOSTRAS_POR_PICO, "n_bins": N_BINS,
        "periodo_pico_grade_dias": periodo_pico_ls,
        "periodo_candidato_dias": periodo_candidato, "periodo_adotado_dias": periodo_adotado,
        "justificativa_periodo": JUSTIFICATIVA_PERIODO,
        "fap_maior_pico_grade": fap_pico, "metodo_fap": "baluev",
        "n_termos_maximo": N_TERMOS_MAXIMO, "n_termos_escolhido": melhor_ajuste["K"],
        "coeficientes_fourier": melhor_ajuste["coeficientes"].tolist(),
        "convencao_fourier": "a0, a1_seno, b1_cosseno, a2_seno, b2_cosseno, ...",
        "periodo_referencia_dias": PERIODO_REFERENCIA, "tipo_catalogo": TIPO_CATALOGO,
        "mag_mais_brilhante_ref": MAG_MAIS_BRILHANTE_REF,
        "mag_menos_brilhante_ref": MAG_MENOS_BRILHANTE_REF,
        "banda_referencia": BANDA_REFERENCIA,
        "url_referencia": URL_REFERENCIA, "data_consulta": DATA_CONSULTA,
        "comparacao_catalogo_preenchida": bool(PERIODO_REFERENCIA is not None
                                               and URL_REFERENCIA and DATA_CONSULTA),
        "python": sys.version.split()[0],
        "bibliotecas": {nome: version(nome) for nome in
                        ["numpy", "pandas", "matplotlib", "scipy", "astropy"]}
    }
    (pasta_saida / "parametros.json").write_text(
        json.dumps(registro, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    print("Resultados salvos em:", pasta_saida.resolve())
else:
    print("Nenhum resultado foi gravado. Ative SALVAR_RESULTADOS ao terminar a investigação.")
